# Cobblemon Atlas｜烈空坐精確 Poser 試渲染

修復版工具會優先選擇 `rayquaza.json`，避免普通烈空坐意外使用 `megarayquaza.json`。此 Notebook 僅渲染 `0384_rayquaza/ground_walk`，輸出到 `reports/semantic-audit/candidates/`，**不會覆蓋**既有的 `renders/`、checkpoint 或原有 1,170 張 WebP。這只是診斷候選，不會自動標為最終通過。

In [ ]:
from pathlib import Path
import os,sys,shutil,subprocess,json,zipfile,time
DRIVE_ROOT=Path('/content/drive/MyDrive/Cobblemon_Atlas')
WORK_ROOT=Path('/content/cobblemon-bedrock')
SITE_ROOT=Path('/content/cobblemon-atlas')
BRANCH='feat/animation-manifest-scanner-phase1'
FORMS=['0384_rayquaza']
ACTIONS=['ground_walk']
FPS,SECONDS,SIZE,SS=12,2,224,2
print('Isolated Rayquaza test:',FORMS,ACTIONS)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert DRIVE_ROOT.is_dir(), f'找不到雲端資料夾 {DRIVE_ROOT}；請確認使用正確 Google 帳號'
for name in ['source_packs','manifests','renders','reports','checkpoints']:
    assert (DRIVE_ROOT/name).is_dir(), f'缺少 Drive 資料夾：{name}'
print('Google Drive 目錄確認成功:', DRIVE_ROOT)

In [ ]:
if not (SITE_ROOT/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',BRANCH,'https://github.com/hwh0410/cobblemon-atlas.git',str(SITE_ROOT)],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'sparse-checkout','set','tools','docs','notebooks','mega-showdown/data'],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'pull','--ff-only','origin',BRANCH],check=True)
sha=subprocess.check_output(['git','-C',str(SITE_ROOT),'rev-parse','--short','HEAD'],text=True).strip()
renderer_source=(SITE_ROOT/'tools/render_action_jobs.py').read_text()
assert 'def choose_actions(' in renderer_source, '目前載入的是舊版選擇器，請重新開啟 GitHub 最新 Notebook'
print('已載入最新版候選來源選擇器，Git commit:',sha)

In [ ]:
# 使用本機暫存磁碟渲染，避免每一幀反覆從 Drive 讀取大量小檔案。
WORK_ROOT.mkdir(parents=True,exist_ok=True)
packs = DRIVE_ROOT/'source_packs'
required = [packs/'development_resource_packs.zip', packs/'merged_pack.zip', packs/'render_portraits.py', DRIVE_ROOT/'manifests/animation-actions.json']
missing = [str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('請先上傳以下來源檔：\n'+'\n'.join(missing))
for archive in required[:2]:
    expected = WORK_ROOT/archive.stem
    if expected.is_dir():
        print('Reuse extracted pack:', expected)
        continue
    print('Extracting:', archive.name, flush=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            dest = (WORK_ROOT/item.filename).resolve()
            if not dest.is_relative_to(WORK_ROOT.resolve()): raise ValueError('Unsafe archive path')
        z.extractall(WORK_ROOT)
    assert expected.is_dir(), f'壓縮檔最外層不是 {archive.stem}，請重新封裝'
(WORK_ROOT/'tools').mkdir(exist_ok=True)
shutil.copy2(packs/'render_portraits.py', WORK_ROOT/'tools/render_portraits.py')
manifest = DRIVE_ROOT/'manifests/animation-actions.json'
data=json.loads(manifest.read_text())
print('Manifest:',data['summary'])
print('Resources prepared locally')

In [ ]:
# 檢查精確 Poser 排序，並將選擇與來源寫入獨立報告。
import importlib.util
tool_path=SITE_ROOT/'tools/render_action_jobs.py'
spec=importlib.util.spec_from_file_location('atlas_rayquaza_jobs',tool_path)
mod=importlib.util.module_from_spec(spec);spec.loader.exec_module(mod)
posers=['bedrock/pokemon/posers/0384_rayquaza/megarayquaza.json',
        'bedrock/pokemon/posers/0384_rayquaza/rayquaza.json']
assert Path(mod.ranked_poser_paths(posers,FORMS[0])[0]).stem=='rayquaza', 'GitHub 尚未取得修正版 Poser 選擇器'
selected=mod.choose_actions(data['entries'],set(FORMS),ACTIONS)
assert len(selected)==1
e,a,action=selected[0]
assert a['model_bones_matched']>0 and a['dynamic_bones']>0
print('Selected clip:',a['clip_key'],a['source_pack'])
print('Ranked posers:',mod.ranked_poser_paths(a.get('poser_paths',[]),FORMS[0]))
report_root=DRIVE_ROOT/'reports/semantic-audit'
report_root.mkdir(parents=True,exist_ok=True)
(report_root/'source-selection-rayquaza-fix.json').write_text(json.dumps({
  'source_pack':a['source_pack'],'clip_key':a['clip_key'],'poser_order':mod.ranked_poser_paths(a.get('poser_paths',[]),FORMS[0])},ensure_ascii=False,indent=2))
subprocess.run([sys.executable,str(tool_path),'--repo',str(WORK_ROOT),'--manifest',str(manifest),'--forms',*FORMS,'--actions',*ACTIONS,'--dry-run'],check=True)

In [ ]:
# 使用獨立本機及 Drive 候選目錄，原先兩片 WebP 和正式 checkpoint 均保持不動。
out=WORK_ROOT/'captures/atlas_rayquaza_poser_fix'
cmd=[sys.executable,str(tool_path),'--repo',str(WORK_ROOT),'--manifest',str(manifest),
     '--forms',*FORMS,'--actions',*ACTIONS,'--fps',str(FPS),'--seconds',str(SECONDS),
     '--size',str(SIZE),'--ss',str(SS),'--out',str(out),'--auto-frame-attempts','8']
result=subprocess.run(cmd,check=False)
print('Renderer return code:',result.returncode)
candidate_dir=DRIVE_ROOT/'reports/semantic-audit/candidates'
candidate_dir.mkdir(parents=True,exist_ok=True)
shutil.copy2(out/'render-jobs.json',candidate_dir/'rayquaza-posers-render-jobs.json')
candidate=out/'0384_rayquaza/ground_walk.webp'
if candidate.is_file():
    shutil.copy2(candidate,candidate_dir/'0384_rayquaza_ground_walk_fixed_poser.webp')
    print('Candidate synced:',candidate_dir/'0384_rayquaza_ground_walk_fixed_poser.webp')
else:
    print('No valid candidate WebP; inspect separate job report.')